# Single-Column ParFlow-CLM Simulation (snow example)

Run a single-column PF-CLM simulation with the `clm_snow_improved` physics defaults:
- **Fractional snow cover**: `FracSnoScheme="CLM"` with `FracSnoRoughness=1e-8`
- **Albedo decay**: slow aging (`AlbedoDecayVis=0.3`, `AlbedoDecayNir=0.1`)
- **Fresh snow albedo**: tuned high (`AlbedoVisNew=0.97`, `AlbedoNirNew=0.67`)
- **Shrubland land cover** (IGBP 7): snow-adapted canopy radiation per `snow_model_sensitivity`

This notebook uses the forcing and CLM inputs prepared in Notebook 1.
Output is written as NetCDF for analysis in Notebook 3.

**Requires:** ParFlow master (commit containing PR #695 "Feature: CLM snow parameterization options" merged 2026-01-28 and PR #709 "SZA-modulated fSCA" merged 2026-03-11) or later.


In [ ]:
import os
import sys
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from datetime import datetime

sys.path.insert(0, os.path.abspath(".."))
from helpers import read_site_config

os.environ["PARFLOW_DIR"] = os.environ.get(
    "PARFLOW_DIR", str(Path.home() / "parflow" / "parflow_21_Jan_26"))
print(f"PARFLOW_DIR: {os.environ['PARFLOW_DIR']}")

from parflow import Run


In [ ]:
# ============================================================
# Load Site Configuration from Notebook 1
# ============================================================
static_write_dir = "../runs/snow/Berthoud_Summit_WY2024"

run_dir = Path(static_write_dir)
cfg = read_site_config(run_dir / "site_config.txt")

site_name  = cfg["site_name"]
water_year = int(cfg["water_year"])
start      = cfg["start"]
end        = cfg["end"]
lat        = float(cfg["lat"])
lon        = float(cfg["lon"])

# Soil (top 2m)
soil_ksat     = float(cfg["soil_ksat"])
soil_porosity = float(cfg["soil_porosity"])
soil_vg_alpha = float(cfg["soil_vg_alpha"])
soil_vg_n     = float(cfg["soil_vg_n"])
soil_sres     = float(cfg["soil_sres"])

# Geology (bottom 6m)
geol_ksat     = float(cfg["geol_ksat"])
geol_porosity = float(cfg["geol_porosity"])
geol_vg_alpha = float(cfg["geol_vg_alpha"])
geol_vg_n     = float(cfg["geol_vg_n"])
geol_sres     = float(cfg["geol_sres"])

wtd_m            = float(cfg["wtd_m"])
forcing_filename = cfg["forcing_file"]

# ---- Override any config values here if needed ----
# wtd_m = -8.0

dt_start = datetime.strptime(start, "%Y-%m-%d")
dt_end   = datetime.strptime(end,   "%Y-%m-%d")
n_hours  = int((dt_end - dt_start).total_seconds() / 3600)

print(f"Site: {site_name} WY{water_year}, {n_hours} hours")
print(f"  Soil:    Ksat={soil_ksat}, porosity={soil_porosity}")
print(f"  Geology: Ksat={geol_ksat}, porosity={geol_porosity}")
print(f"  WTD BC:  {wtd_m} m")


In [ ]:
# ============================================================
# ParFlow-CLM Run Setup (snow example)
# ============================================================
# Subsurface geometry is the same 8m / 10-layer column as the ET example.
# Snow-specific CLM keys are in the ----- CLM SNOW PHYSICS ----- section.

run_name = "pfclm_sc"
run = Run(run_name)
run.FileVersion = 4

# ----- TIMING -----
run.TimingInfo.BaseUnit     = 1.0
run.TimingInfo.StartCount   = 0
run.TimingInfo.StartTime    = 0.0
run.TimingInfo.StopTime     = n_hours
run.TimingInfo.DumpInterval = 1.0
run.TimeStep.Type           = "Constant"
run.TimeStep.Value          = 1.0

# ----- GRID (single column, 10 layers, 8m deep) -----
run.Process.Topology.P = 1
run.Process.Topology.Q = 1
run.Process.Topology.R = 1

run.ComputationalGrid.Lower.X = 0.0
run.ComputationalGrid.Lower.Y = 0.0
run.ComputationalGrid.Lower.Z = 0.0
run.ComputationalGrid.DX      = 2.0
run.ComputationalGrid.DY      = 2.0
run.ComputationalGrid.DZ      = 1.0
run.ComputationalGrid.NX      = 1
run.ComputationalGrid.NY      = 1
run.ComputationalGrid.NZ      = 10

# ----- DOMAIN + TWO-LAYER GEOMETRY -----
run.GeomInput.Names = "domain_input soil_input geol_input"

run.GeomInput.domain_input.InputType = "Box"
run.GeomInput.domain_input.GeomName  = "domain"
run.Geom.domain.Lower.X = 0.0
run.Geom.domain.Lower.Y = 0.0
run.Geom.domain.Lower.Z = 0.0
run.Geom.domain.Upper.X = 2.0
run.Geom.domain.Upper.Y = 2.0
run.Geom.domain.Upper.Z = 10.0
run.Geom.domain.Patches = "x_lower x_upper y_lower y_upper z_lower z_upper"

run.GeomInput.soil_input.InputType = "Box"
run.GeomInput.soil_input.GeomName  = "soil"
run.Geom.soil.Lower.X = 0.0
run.Geom.soil.Lower.Y = 0.0
run.Geom.soil.Lower.Z = 6.0
run.Geom.soil.Upper.X = 2.0
run.Geom.soil.Upper.Y = 2.0
run.Geom.soil.Upper.Z = 10.0

run.GeomInput.geol_input.InputType = "Box"
run.GeomInput.geol_input.GeomName  = "geology"
run.Geom.geology.Lower.X = 0.0
run.Geom.geology.Lower.Y = 0.0
run.Geom.geology.Lower.Z = 0.0
run.Geom.geology.Upper.X = 2.0
run.Geom.geology.Upper.Y = 2.0
run.Geom.geology.Upper.Z = 6.0

# ----- VARIABLE DZ -----
run.Solver.Nonlinear.VariableDz = True
run.dzScale.GeomNames           = "domain"
run.dzScale.Type                = "nzList"
run.dzScale.nzListNumber        = 10
run.Cell._0.dzScale.Value = 1.0
run.Cell._1.dzScale.Value = 1.0
run.Cell._2.dzScale.Value = 1.0
run.Cell._3.dzScale.Value = 1.0
run.Cell._4.dzScale.Value = 1.0
run.Cell._5.dzScale.Value = 1.0
run.Cell._6.dzScale.Value = 1.0
run.Cell._7.dzScale.Value = 0.6
run.Cell._8.dzScale.Value = 0.3
run.Cell._9.dzScale.Value = 0.1

# ----- HYDRAULIC PROPERTIES (two-layer) -----
run.Geom.Perm.Names          = "soil geology"
run.Geom.soil.Perm.Type      = "Constant"
run.Geom.soil.Perm.Value     = soil_ksat
run.Geom.geology.Perm.Type   = "Constant"
run.Geom.geology.Perm.Value  = geol_ksat

run.Perm.TensorType              = "TensorByGeom"
run.Geom.Perm.TensorByGeom.Names = "domain"
run.Geom.domain.Perm.TensorValX  = 1.0
run.Geom.domain.Perm.TensorValY  = 1.0
run.Geom.domain.Perm.TensorValZ  = 1.0

run.SpecificStorage.Type               = "Constant"
run.SpecificStorage.GeomNames          = "domain"
run.Geom.domain.SpecificStorage.Value  = 1.0e-4

run.Geom.Porosity.GeomNames     = "soil geology"
run.Geom.soil.Porosity.Type     = "Constant"
run.Geom.soil.Porosity.Value    = soil_porosity
run.Geom.geology.Porosity.Type  = "Constant"
run.Geom.geology.Porosity.Value = geol_porosity

# ----- PHASES -----
run.Phase.Names                 = "water"
run.Phase.water.Density.Type    = "Constant"
run.Phase.water.Density.Value   = 1.0
run.Phase.water.Viscosity.Type  = "Constant"
run.Phase.water.Viscosity.Value = 1.0
run.Phase.water.Mobility.Type   = "Constant"
run.Phase.water.Mobility.Value  = 1.0

run.Phase.RelPerm.Type         = "VanGenuchten"
run.Phase.RelPerm.GeomNames    = "soil geology"
run.Geom.soil.RelPerm.Alpha    = soil_vg_alpha
run.Geom.soil.RelPerm.N        = soil_vg_n
run.Geom.geology.RelPerm.Alpha = geol_vg_alpha
run.Geom.geology.RelPerm.N     = geol_vg_n

run.Phase.Saturation.Type          = "VanGenuchten"
run.Phase.Saturation.GeomNames     = "soil geology"
run.Geom.soil.Saturation.Alpha     = soil_vg_alpha
run.Geom.soil.Saturation.N         = soil_vg_n
run.Geom.soil.Saturation.SRes      = soil_sres
run.Geom.soil.Saturation.SSat      = 1.0
run.Geom.geology.Saturation.Alpha  = geol_vg_alpha
run.Geom.geology.Saturation.N      = geol_vg_n
run.Geom.geology.Saturation.SRes   = geol_sres
run.Geom.geology.Saturation.SSat   = 1.0

# ----- MISC -----
run.Contaminants.Names = ""
run.Gravity            = 1.0
run.Domain.GeomName    = "domain"
run.Wells.Names        = ""
run.KnownSolution      = "NoKnownSolution"

# ----- TIME CYCLES -----
run.Cycle.Names                    = "constant"
run.Cycle.constant.Names           = "alltime"
run.Cycle.constant.alltime.Length  = 1
run.Cycle.constant.Repeat          = -1

# ----- BOUNDARY CONDITIONS -----
run.BCPressure.PatchNames = "x_lower x_upper y_lower y_upper z_lower z_upper"

run.Patch.x_lower.BCPressure.Type          = "FluxConst"
run.Patch.x_lower.BCPressure.Cycle         = "constant"
run.Patch.x_lower.BCPressure.alltime.Value = 0.0
run.Patch.y_lower.BCPressure.Type          = "FluxConst"
run.Patch.y_lower.BCPressure.Cycle         = "constant"
run.Patch.y_lower.BCPressure.alltime.Value = 0.0

run.Patch.z_lower.BCPressure.Type          = "DirEquilRefPatch"
run.Patch.z_lower.BCPressure.RefGeom       = "domain"
run.Patch.z_lower.BCPressure.RefPatch      = "z_upper"
run.Patch.z_lower.BCPressure.Cycle         = "constant"
run.Patch.z_lower.BCPressure.alltime.Value = wtd_m

run.Patch.x_upper.BCPressure.Type          = "FluxConst"
run.Patch.x_upper.BCPressure.Cycle         = "constant"
run.Patch.x_upper.BCPressure.alltime.Value = 0.0
run.Patch.y_upper.BCPressure.Type          = "FluxConst"
run.Patch.y_upper.BCPressure.Cycle         = "constant"
run.Patch.y_upper.BCPressure.alltime.Value = 0.0
run.Patch.z_upper.BCPressure.Type          = "OverlandFlow"
run.Patch.z_upper.BCPressure.Cycle         = "constant"
run.Patch.z_upper.BCPressure.alltime.Value = 0.0

# ----- SLOPES -----
run.TopoSlopesX.Type              = "Constant"
run.TopoSlopesX.GeomNames         = "domain"
run.TopoSlopesX.Geom.domain.Value = 0.1
run.TopoSlopesY.Type              = "Constant"
run.TopoSlopesY.GeomNames         = "domain"
run.TopoSlopesY.Geom.domain.Value = 0.0
run.Mannings.Type                 = "Constant"
run.Mannings.GeomNames            = "domain"
run.Mannings.Geom.domain.Value    = 2.e-6
run.PhaseSources.water.Type       = "Constant"
run.PhaseSources.water.GeomNames  = "domain"
run.PhaseSources.water.Geom.domain.Value = 0.0

# ----- SOLVER -----
run.Solver         = "Richards"
run.Solver.MaxIter = 9000
run.Solver.Nonlinear.MaxIter           = 100
run.Solver.Nonlinear.ResidualTol       = 1e-7
run.Solver.Nonlinear.EtaChoice         = "EtaConstant"
run.Solver.Nonlinear.EtaValue          = 0.01
run.Solver.Nonlinear.UseJacobian       = True
run.Solver.Nonlinear.DerivativeEpsilon = 1e-12
run.Solver.Nonlinear.StepTol           = 1e-15
run.Solver.Nonlinear.Globalization     = "LineSearch"
run.Solver.Linear.KrylovDimension      = 100
run.Solver.Linear.MaxRestarts          = 5
run.Solver.Linear.Preconditioner       = "PFMG"
run.Solver.PrintSubsurf                = False
run.Solver.Drop                        = 1E-20
run.Solver.AbsTol                      = 1E-9

# ----- OUTPUT: PFB (disabled) -----
run.Solver.PrintSubsurfData     = False
run.Solver.PrintPressure        = False
run.Solver.PrintSaturation      = False
run.Solver.PrintCLM             = False
run.Solver.PrintMask            = False
run.Solver.PrintSpecificStorage = False
run.Solver.PrintEvapTrans       = False
run.Solver.WriteSiloMannings    = False
run.Solver.WriteSiloMask        = False
run.Solver.WriteSiloSlopes      = False
run.Solver.WriteSiloSaturation  = False

# ----- OUTPUT: NetCDF -----
run.NetCDF.NumStepsPerFile     = n_hours
run.NetCDF.WritePressure       = True
run.NetCDF.WriteSubsurface     = False
run.NetCDF.WriteSaturation     = True
run.NetCDF.WriteCLM            = True
run.NetCDF.CLMNumStepsPerFile  = n_hours

# ----- CLM CORE CONFIGURATION -----
run.Solver.LSM             = "CLM"
run.Solver.CLM.MetForcing  = "1D"
run.Solver.CLM.MetFileName = forcing_filename
run.Solver.CLM.MetFilePath = "."

run.Solver.CLM.EvapBeta       = "Linear"
run.Solver.CLM.VegWaterStress = "Saturation"
run.Solver.CLM.ResSat         = 0.2
run.Solver.CLM.WiltingPoint   = 0.2
run.Solver.CLM.FieldCapacity  = 1.0
run.Solver.CLM.IrrigationType = "none"
run.Solver.CLM.RZWaterStress  = 1
run.Solver.CLM.RootZoneNZ     = 5
run.Solver.CLM.SoiLayer       = 4

# ----- CLM SNOW PHYSICS (clm_snow_improved) -----
# Fractional snow cover: CLM default scheme with minimal roughness
run.Solver.CLM.FracSnoScheme     = "CLM"
run.Solver.CLM.FracSnoRoughness  = 1e-8

# Albedo — slow aging + tuned fresh-snow values (best low-bias result:
# |bias|=13.7%, melt-out +7d, NSE=0.87 in the shrub+combined study)
run.Solver.CLM.AlbedoDecayVis = 0.3
run.Solver.CLM.AlbedoDecayNir = 0.1
run.Solver.CLM.AlbedoVisNew   = 0.97
run.Solver.CLM.AlbedoNirNew   = 0.67

# --- Optional: switch to SZA-modulated fSCA (PR #709) ---
# Uncomment to replace FracSnoScheme=CLM with the solar-zenith-angle
# scheme. This is the "interpolating" formulation that transitions
# between bare-soil and snow roughness based on sun angle.
# run.Solver.CLM.FracSnoScheme       = "SZA"
# run.Solver.CLM.FracSnoRoughnessMin = 1e-8
# run.Solver.CLM.FracSnoRoughnessMax = 0.2
# run.Solver.CLM.FracSnoGammaSZA     = 4.0
# run.Solver.CLM.FracSnoAvgWindow    = 72.0

# ----- CLM OUTPUT -----
run.Solver.PrintLSMSink         = False
run.Solver.CLM.CLMDumpInterval  = 1
run.Solver.CLM.CLMFileDir       = "output/"
run.Solver.CLM.BinaryOutDir     = False
run.Solver.CLM.IstepStart       = 1
run.Solver.WriteCLMBinary       = False
run.Solver.WriteSiloCLM         = False
run.Solver.CLM.WriteLogs        = False
run.Solver.CLM.WriteLastRST     = True
run.Solver.CLM.DailyRST         = False
run.Solver.CLM.SingleFile       = True

# ----- INITIAL CONDITIONS -----
run.ICPressure.Type                 = "HydroStaticPatch"
run.ICPressure.GeomNames            = "domain"
run.Geom.domain.ICPressure.Value    = wtd_m
run.Geom.domain.ICPressure.RefGeom  = "domain"
run.Geom.domain.ICPressure.RefPatch = "z_lower"

print("ParFlow run configured (snow):")
print(f"  Subsurface: soil Ksat={soil_ksat}, geol Ksat={geol_ksat}, WTD={wtd_m} m")
print(f"  Snow:       FracSno=CLM (z0=1e-8), Albedo slow-decay + high-fresh")
print(f"  Land cover: IGBP 7 (Open Shrubland)")


In [ ]:
# Run ParFlow
run.run(working_directory=static_write_dir, skip_validation=True)


## Quick-Look Output Verification

Load the NetCDF output and plot SWE + snow depth to verify the run completed successfully.


In [ ]:
# Load CLM output
run_dir  = Path(static_write_dir)
clm_files = sorted(run_dir.glob("*.out.CLM.*.nc"))

if not clm_files:
    raise FileNotFoundError(f"No CLM NetCDF output found in {run_dir}")

clm_ds = xr.open_dataset(clm_files[0])
n_steps = clm_ds.sizes.get("time", 0)

if n_steps < n_hours:
    print(f"WARNING: Run incomplete! Got {n_steps}/{n_hours} timesteps "
          f"({n_steps/n_hours*100:.0f}%). Check pfclm_sc.out.log for solver failures.")
else:
    print(f"Run complete: {n_steps} timesteps")

print(f"CLM variables: {list(clm_ds.data_vars)}")
if "swe_out" in clm_ds:
    swe = clm_ds["swe_out"].values.flatten()
    print(f"  SWE range: {swe.min():.1f} to {swe.max():.1f} mm, peak={swe.max():.1f} mm")


In [ ]:
# Plot SWE and snow depth
dates = pd.date_range(start, periods=n_steps, freq="h")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

if "swe_out" in clm_ds:
    axes[0].plot(dates, clm_ds["swe_out"].values.flatten(),
                 lw=0.5, color="steelblue")
    axes[0].set_ylabel("SWE [mm]")
    axes[0].set_title(f"{site_name} WY{water_year} — PF-CLM Snow Output "
                      f"({n_steps}/{n_hours} hrs)")
    axes[0].grid(alpha=0.3)

# Snow depth (snowdp) if available
snowdp_var = None
for v in ["snowdp", "SnowDepth", "snow_depth"]:
    if v in clm_ds:
        snowdp_var = v
        break
if snowdp_var is not None:
    axes[1].plot(dates, clm_ds[snowdp_var].values.flatten(),
                 lw=0.5, color="navy")
    axes[1].set_ylabel(f"Snow depth [m] ({snowdp_var})")
else:
    axes[1].text(0.5, 0.5, "No snow depth variable in output",
                 transform=axes[1].transAxes, ha="center", va="center")

axes[1].set_xlabel("Date")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

clm_ds.close()
